# nano_climb, as a notebook

The smallest complete hill climb, one step per cell: a problem (what counts as better), a selector
policy (which attempt to build on), an operator policy (what to do with it), two operators (what to
ask the coding agent), a budget, and the search. Claude Code writes the solutions, so the search
cell takes a few minutes and costs a little.

Run it from a hillclimb dir (`hillclimb init` makes one). The script version is `nano_climb.py`.

In [ ]:
AGENT = "claude-code"   # who writes the solutions; "dummy" runs the machinery with no model calls

## 1. The problem

A solution is a script with ten seconds to find a prime and write it to `prime.txt`. The score is
how many digits the prime has, and the scorer checks it really is prime (Miller-Rabin).

The scorer goes in a file of its own: the verifier imports it in a separate process, which can't see
functions defined in a notebook cell.

In [ ]:
%%writefile prime_score.py
import random
from pathlib import Path


def is_probably_prime(n: int, rounds: int = 24) -> bool:
    if n < 2:
        return False
    for p in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37):
        if n % p == 0:
            return n == p
    d, s = n - 1, 0
    while d % 2 == 0:
        d, s = d // 2, s + 1
    rng = random.Random(0)
    for _ in range(rounds):
        x = pow(rng.randrange(2, n - 1), d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = x * x % n
            if x == n - 1:
                break
        else:
            return False
    return True


def prime_digits(run_dir: Path) -> float:
    """How many digits the prime in prime.txt has; 0 if it is not a prime."""
    text = (run_dir / "prime.txt").read_text().strip()
    if not text.isdigit() or len(text) > 20_000:
        return 0.0
    return float(len(text)) if is_probably_prime(int(text)) else 0.0

In [ ]:
from hillclimb import Budget, Climber, Problem
from hillclimb.sdk import Action, Attempt, Operator, OperatorContext, OperatorPolicy, SearchState, Selection, SelectorPolicy
from prime_score import prime_digits

problem = Problem(
    "largest-prime",
    score=prime_digits,
    higher_is_better=True,
    description=(
        "Find the largest prime number you can and write it, in decimal, to prime.txt. Pure\n"
        "Python with the standard library only; no hard-coded primes. The verifier checks primality\n"
        "with Miller-Rabin, so the number must really be prime. At most 20000 digits."
    ),
    output="prime.txt",
    time_limit_s=10,      # the verifier stops a solution that runs longer: a failed attempt
    baseline=0.0,
)

## 2. The selector policy: which attempt does the next one start from?

The whole schedule is here: a failing attempt is revised first, nothing is built on until
`num_drafts` fresh attempts exist, then the best so far is built on.

In [ ]:
class BestSoFar(SelectorPolicy):
    name = "best-so-far"
    DEFAULTS = {"num_drafts": 2}  # its one knob: BestSoFar(num_drafts=...)

    def schedule(self, state: SearchState, *, busy=frozenset()) -> Selection | None:
        failing = [c for c in state.journal.candidates.values() if c.status in ("failing", "buggy")]
        if failing and not state.journal.children(failing[-1].candidate_id):
            return Selection(failing[-1].candidate_id)  # revise the newest failing attempt first
        if len(state.journal.drafts()) < int(self.param("num_drafts")):
            return None  # a root step: write a fresh attempt
        return self.select(state, busy=busy)

    def select(self, state: SearchState, *, busy=frozenset()) -> Selection | None:
        scored = state.journal.scored_candidates()
        if not scored:
            return None
        best = max(scored, key=lambda c: c.val_score)
        return Selection(best.candidate_id)

## 3. The operator policy: what to do with it

Nothing selected means write a fresh attempt; an attempt selected means revise it.

In [ ]:
class WriteOrRevise(OperatorPolicy):
    name = "write-or-revise"

    def propose(self, state: SearchState, selection: Selection | None) -> Action | None:
        if selection is None:
            return Action("write")
        return Action("revise", target_id=selection.target_id)

## 4. The operators: how one attempt is made

Each operator is a prompt. `copy_parent` puts the selected attempt's `solution.py` in the working
directory to edit, and the harness fills in the problem's contract where `{{contract}}` stands.

In [ ]:
class Write(Operator):
    name, kind = "write", "create"

    def prepare(self, ctx: OperatorContext) -> Attempt:
        prompt = (
            "Write solution.py from scratch for the problem below. Think about where the time\n"
            "goes: finding a candidate, and proving it prime. Say in notes.md what you chose.\n\n"
            "{{contract}}"
        )
        return Attempt(prompt=prompt)


class Revise(Operator):
    name, kind, needs_target = "revise", "refine", True

    def prepare(self, ctx: OperatorContext) -> Attempt:
        node = ctx.target
        if node.val_score is None:
            what = "It failed, or ran past the time limit. Fix that first, then make it find a bigger prime."
        else:
            what = f"It found a {node.val_score:g}-digit prime. Make it find a bigger one within the time limit."
        prompt = (
            f"solution.py in this directory is your previous attempt. {what}\n"
            "Keep the primality test sound; the verifier re-checks it. Say in notes.md what you changed.\n\n"
            "{{contract}}"
        )
        return Attempt(prompt=prompt, copy_parent=True)

## 5. The budget, the climber, the search

Eight attempts or 45 minutes, whichever comes first. The log below shows every attempt as it lands.

In [ ]:
budget = Budget(wall_clock="45m", evaluations=8)
climber = Climber(
    selector_policy=BestSoFar(num_drafts=2),   # two fresh attempts, then build on the best
    operator_policy=WriteOrRevise(),           # write, or revise
    operators=[Write(), Revise()],             # the two prompts above
)

climber.search(problem, budget=budget, agent=AGENT, learning=False);   # the ; keeps the climber's repr out of the output

## 6. What it found

In [ ]:
climber.to_frame()[["parent_id", "operator", "status", "val_score"]]

In [ ]:
for step in climber.history:   # each time the best went up
    print(f"{step.candidate_id}  {step.score:6g} digits  at {step.minutes:.1f} min")

In [ ]:
print(climber.solution or "nothing beat the baseline")   # the winning solution.py